In [1]:
!pip install tensorflow  

In [7]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from glob import glob
from PIL import Image

# --- CONFIGURATION ---
data_root = r"C:\Users\ashut\Desktop\ElementaryCQT"
image_size = (200, 200)
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE

# --- STEP 1: Load All Images and Labels ---
def load_data():
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(data_root))
    class_map = {name: idx for idx, name in enumerate(class_names)}

    for shape_class in class_names:
        shape_dir = os.path.join(data_root, shape_class)
        for subtype in os.listdir(shape_dir):
            subtype_dir = os.path.join(shape_dir, subtype)
            if os.path.isdir(subtype_dir):
                for img_file in glob(f"{subtype_dir}/*.png") + glob(f"{subtype_dir}/*.jpg"):
                    image_paths.append(img_file)
                    labels.append(class_map[shape_class])

    return image_paths, labels, class_names

# --- STEP 2: Preprocess Images ---
def preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])  # Set shape manually
    image = tf.image.rgb_to_grayscale(image)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# --- STEP 3: Create Dataset Splits ---
image_paths, labels, class_names = load_data()
num_classes = len(class_names)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

def make_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)

print(f"Dataset sizes — Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

# --- STEP 4: Build Geo-CNN Model ---
class EdgeLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.edge_conv = tf.keras.layers.Conv2D(8, kernel_size=3, padding='same', activation='relu')

    def call(self, x):
        return self.edge_conv(x)

class ShapeAttention(tf.keras.layers.Layer):
    def call(self, x):
        # x: (B, H, W, C)
        attn = tf.reduce_mean(x, axis=-1, keepdims=True)  # (B, H, W, 1)
        flat_attn = tf.reshape(attn, [tf.shape(x)[0], -1])  # (B, H*W)
        norm_attn = tf.nn.softmax(flat_attn, axis=-1)  # attention over spatial locations
        norm_attn = tf.reshape(norm_attn, tf.shape(attn))  # back to (B, H, W, 1)
        return x * norm_attn  # apply attention mask to original input



class GeometricEmbedding(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.pool = tf.keras.layers.GlobalAveragePooling2D()
        self.dense = tf.keras.layers.Dense(32, activation='relu')

    def call(self, x):
        return self.dense(self.pool(x))

def build_geo_cnn(input_shape=(200, 200, 1), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)
    x = EdgeLayer()(inputs)
    x = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = ShapeAttention()(x)
    x = GeometricEmbedding()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)

# --- STEP 5: Compile and Train ---
model = build_geo_cnn(num_classes=num_classes)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_ds, validation_data=val_ds, epochs=10)

# --- STEP 6: Evaluate on Test Set ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"✅ Test Accuracy: {test_acc:.4f}")


Dataset sizes — Train: 304000, Val: 38000, Test: 38000
Epoch 1/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 984s 103ms/step - accuracy: 0.2809 - loss: 1.8168 - val_accuracy: 0.5465 - val_loss: 1.1388
Epoch 2/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 919s 97ms/step - accuracy: 0.6588 - loss: 0.8554 - val_accuracy: 0.7423 - val_loss: 0.6115
Epoch 3/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 915s 96ms/step - accuracy: 0.7649 - loss: 0.5801 - val_accuracy: 0.7542 - val_loss: 0.6221
Epoch 4/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 954s 100ms/step - accuracy: 0.7913 - loss: 0.5052 - val_accuracy: 0.8083 - val_loss: 0.4585
Epoch 5/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 915s 96ms/step - accuracy: 0.8084 - loss: 0.4599 - val_accuracy: 0.8201 - val_loss: 0.4294
Epoch 6/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 994s 105ms/step - accuracy: 0.8199 - loss: 0.4277 - val_accuracy: 0.8171 - val_loss: 0.4345
Epoch 7/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 950s 100ms/step - accuracy: 0.8325 - loss: 0.3996 - val_accuracy: 0.8563 - val_loss: 0.3531
Epoch 8/10

CUSTOM CNN MODEL VERSION 2 

INCLUDING CUSTOM ATTENTION LAYERS AND CONV LAYER OF 2 AND ATTENTION LAYER OF 2

In [ ]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from glob import glob
from PIL import Image

# --- CONFIGURATION ---
data_root = r"C:\Users\ashut\Desktop\ElementaryCQT"
image_size = (200, 200)
batch_size = 128
AUTOTUNE = tf.data.AUTOTUNE

# --- Enable mixed precision if GPU supports ---
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# --- STEP 1: Load All Images and Labels ---
def load_data():
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(data_root))
    class_map = {name: idx for idx, name in enumerate(class_names)}

    for shape_class in class_names:
        shape_dir = os.path.join(data_root, shape_class)
        for subtype in os.listdir(shape_dir):
            subtype_dir = os.path.join(shape_dir, subtype)
            if os.path.isdir(subtype_dir):
                for img_file in glob(f"{subtype_dir}/*.png") + glob(f"{subtype_dir}/*.jpg"):
                    image_paths.append(img_file)
                    labels.append(class_map[shape_class])

    return image_paths, labels, class_names

# --- STEP 2: Preprocess Images ---
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
])

def preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])  # Set shape manually
    image = tf.image.rgb_to_grayscale(image)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32) / 255.0
    image = data_augmentation(image)
    return image, label

# --- STEP 3: Create Dataset Splits ---
image_paths, labels, class_names = load_data()
num_classes = len(class_names)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

def make_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)

print(f"Dataset sizes — Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

# --- STEP 4: Build Improved Geo-CNN Model ---
class EdgeLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.edge_conv = tf.keras.layers.Conv2D(8, kernel_size=3, padding='same', activation='relu')

    def call(self, x):
        return self.edge_conv(x)

class ShapeAttention(tf.keras.layers.Layer):
    def call(self, x):
        attn = tf.reduce_mean(x, axis=-1, keepdims=True)
        flat_attn = tf.reshape(attn, [tf.shape(x)[0], -1])
        norm_attn = tf.nn.softmax(flat_attn, axis=-1)
        norm_attn = tf.reshape(norm_attn, tf.shape(attn))
        return x * norm_attn

class GeometricEmbedding(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.pool = tf.keras.layers.GlobalAveragePooling2D()
        self.dense = tf.keras.layers.Dense(32, activation='relu')

    def call(self, x):
        return self.dense(self.pool(x))

def build_geo_cnn(input_shape=(200, 200, 1), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)
    x = EdgeLayer()(inputs)
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = ShapeAttention()(x)
    x = GeometricEmbedding()(x)

    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32')(x)  # force output float32 for mixed precision

    return tf.keras.Model(inputs, outputs)

# --- STEP 5: Compile and Train ---
initial_lr = 1e-3
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=initial_lr,
    decay_steps=10000,
    decay_rate=0.9
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

model = build_geo_cnn(num_classes=num_classes)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_ds, validation_data=val_ds, epochs=3)

# --- STEP 6: Evaluate on Test Set ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"✅ Test Accuracy: {test_acc:.4f}")

# --- STEP 7: Plot Training Curves ---
import matplotlib.pyplot as plt

def plot_history(history):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()

plot_history(history)


Dataset sizes — Train: 403047, Val: 50381, Test: 50381
Epoch 1/3
   9/3149 ━━━━━━━━━━━━━━━━━━━━ 43:15:36 50s/step - accuracy: 0.5391 - loss: 1.0935

In [ ]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from glob import glob
from PIL import Image

# --- CONFIGURATION ---
data_root = r"C:\Users\ashut\Desktop\ElementaryCQT"
image_size = (200, 200)
batch_size = 64
AUTOTUNE = tf.data.AUTOTUNE

# --- STEP 1: Load All Images and Labels ---
def load_data():
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(data_root))
    class_map = {name: idx for idx, name in enumerate(class_names)}

    for shape_class in class_names:
        shape_dir = os.path.join(data_root, shape_class)
        for subtype in os.listdir(shape_dir):
            subtype_dir = os.path.join(shape_dir, subtype)
            if os.path.isdir(subtype_dir):
                for img_file in glob(f"{subtype_dir}/*.png") + glob(f"{subtype_dir}/*.jpg"):
                    image_paths.append(img_file)
                    labels.append(class_map[shape_class])

    return image_paths, labels, class_names

# --- STEP 2: Preprocess Images ---
def preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])  # Set shape manually
    image = tf.image.rgb_to_grayscale(image)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# --- STEP 3: Create Dataset Splits ---
image_paths, labels, class_names = load_data()
num_classes = len(class_names)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

def make_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)

print(f"Dataset sizes — Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

# --- STEP 4: Build Geo-CNN Model ---
class EdgeLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.edge_conv = tf.keras.layers.Conv2D(8, kernel_size=3, padding='same', activation='relu')

    def call(self, x):
        return self.edge_conv(x)

class ShapeAttention(tf.keras.layers.Layer):
    def call(self, x):
        # x: (B, H, W, C)
        attn = tf.reduce_mean(x, axis=-1, keepdims=True)  # (B, H, W, 1)
        flat_attn = tf.reshape(attn, [tf.shape(x)[0], -1])  # (B, H*W)
        norm_attn = tf.nn.softmax(flat_attn, axis=-1)  # attention over spatial locations
        norm_attn = tf.reshape(norm_attn, tf.shape(attn))  # back to (B, H, W, 1)
        return x * norm_attn  # apply attention mask to original input



class GeometricEmbedding(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.pool = tf.keras.layers.GlobalAveragePooling2D()
        self.dense = tf.keras.layers.Dense(32, activation='relu')

    def call(self, x):
        return self.dense(self.pool(x))

def build_geo_cnn(input_shape=(200, 200, 1), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)
    x = EdgeLayer()(inputs)
    x = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = ShapeAttention()(x)
    x = GeometricEmbedding()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)

# --- STEP 5: Compile and Train ---
model = build_geo_cnn(num_classes=num_classes)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_ds, validation_data=val_ds, epochs=3)

# --- STEP 6: Evaluate on Test Set ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"✅ Test Accuracy: {test_acc:.4f}")


In [ ]:
import os
import tensorflow as tf
import numpy as np
from glob import glob
import matplotlib.pyplot as plt

# --- GPU CONFIGURATION (Commented out) ---

# Check for GPU availability
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

# Configure GPU memory growth to avoid memory errors
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        #Optionally limit GPU memory
        tf.config.experimental.set_virtual_device_configuration(
            gpus[0],
            [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=4096)]
        )
        
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)


# --- CONFIGURATION ---
# Base directory containing train, val, and test folders
data_root = r"C:\Users\ashut\Desktop\ElementaryCQT"
train_dir = os.path.join(data_root, "train")
val_dir = os.path.join(data_root, "val")
test_dir = os.path.join(data_root, "test")

image_size = (200, 200)
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE

# --- STEP 1: Load Images from Train/Val/Test Folders ---
def get_class_names(directory):
    """Get class names from directory structure"""
    return sorted([d for d in os.listdir(directory) 
                  if os.path.isdir(os.path.join(directory, d))])

def load_data_from_directory(directory):
    """Load data from a directory with class subdirectories"""
    image_paths = []
    labels = []
    class_names = get_class_names(directory)
    class_map = {name: idx for idx, name in enumerate(class_names)}
    
    for shape_class in class_names:
        class_dir = os.path.join(directory, shape_class)
        # Check if there are subfolders within each class
        subfolders = [f for f in os.listdir(class_dir) 
                     if os.path.isdir(os.path.join(class_dir, f))]
        
        if subfolders:
            # Case: class/subtype/images.png structure
            for subtype in subfolders:
                subtype_dir = os.path.join(class_dir, subtype)
                for img_file in glob(f"{subtype_dir}/*.png") + glob(f"{subtype_dir}/*.jpg"):
                    image_paths.append(img_file)
                    labels.append(class_map[shape_class])
        else:
            # Case: class/images.png structure (no subtypes)
            for img_file in glob(f"{class_dir}/*.png") + glob(f"{class_dir}/*.jpg"):
                image_paths.append(img_file)
                labels.append(class_map[shape_class])
    
    return image_paths, labels, class_names

# Load data from each directory
print("Loading train data...")
train_paths, train_labels, class_names = load_data_from_directory(train_dir)
print("Loading validation data...")
val_paths, val_labels, _ = load_data_from_directory(val_dir)
print("Loading test data...")
test_paths, test_labels, _ = load_data_from_directory(test_dir)

num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")
print(f"Dataset sizes — Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

# --- STEP 2: Preprocessing ---
def preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])  # Set shape manually
    image = tf.image.rgb_to_grayscale(image)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)

# --- STEP 3: Enhanced Geometric CNN Architecture ---

# Advanced Edge Detection Layer
class AdvancedEdgeLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        # Initialize Sobel kernels
        self.h_kernel = tf.constant([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=tf.float32)
        self.v_kernel = tf.constant([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=tf.float32)
        
        # Learnable edge features
        self.edge_conv = tf.keras.layers.Conv2D(16, kernel_size=3, padding='same', activation='relu')
        self.bn = tf.keras.layers.BatchNormalization()
        
    def call(self, x, training=False):
        # Extract grayscale if input is RGB
        if x.shape[-1] > 1:
            x_gray = tf.image.rgb_to_grayscale(x)
        else:
            x_gray = x
            
        # Reshape kernels for convolution
        h_kernel = tf.reshape(self.h_kernel, [3, 3, 1, 1])
        v_kernel = tf.reshape(self.v_kernel, [3, 3, 1, 1])
        
        # Apply Sobel operators
        h_edges = tf.nn.conv2d(x_gray, h_kernel, strides=[1, 1, 1, 1], padding='SAME')
        v_edges = tf.nn.conv2d(x_gray, v_kernel, strides=[1, 1, 1, 1], padding='SAME')
        
        # Combine edge magnitudes
        edge_magnitude = tf.sqrt(tf.square(h_edges) + tf.square(v_edges))
        
        # Pass through learnable convolutional layer
        edge_features = self.edge_conv(tf.concat([x, edge_magnitude], axis=-1))
        return self.bn(edge_features, training=training)

# Spatial Attention Module
class SpatialAttention(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(32, kernel_size=7, padding='same')
        self.conv2 = tf.keras.layers.Conv2D(16, kernel_size=5, padding='same')
        self.conv3 = tf.keras.layers.Conv2D(1, kernel_size=3, padding='same')
        
    def call(self, x):
        # Channel attention: max and average pooling along channel axis
        avg_pool = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=-1, keepdims=True)
        
        # Concatenate pooled features
        pooled = tf.concat([avg_pool, max_pool], axis=-1)
        
        # Process with convolutional layers
        attn = tf.nn.relu(self.conv1(pooled))
        attn = tf.nn.relu(self.conv2(attn)) 
        attn = self.conv3(attn)
        
        # Apply sigmoid to get attention weights
        attn = tf.sigmoid(attn)
        
        # Apply attention mask to input
        return x * attn

# Residual Block for Better Gradient Flow
class GeometricResidualBlock(tf.keras.layers.Layer):
    def __init__(self, filters):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(filters, 3, padding='same')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(filters, 3, padding='same')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.skip_conn = tf.keras.layers.Conv2D(filters, 1, padding='same')
        
    def call(self, inputs, training=False):
        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        
        # Skip connection
        skip = self.skip_conn(inputs)
        
        return tf.nn.relu(x + skip)

# Enhanced Feature Extractor
class ShapeFeatureExtractor(tf.keras.layers.Layer):
    def __init__(self, embedding_dim=64):
        super().__init__()
        # Enhanced pooling with global max and average
        self.avg_pool = tf.keras.layers.GlobalAveragePooling2D()
        self.max_pool = tf.keras.layers.GlobalMaxPooling2D()
        
        # Feature projections
        self.dense1 = tf.keras.layers.Dense(embedding_dim, activation='relu')
        self.dense2 = tf.keras.layers.Dense(embedding_dim, activation='relu')
        self.dropout = tf.keras.layers.Dropout(0.3)
        
    def call(self, x, training=False):
        # Extract both average and max features
        avg_features = self.avg_pool(x)
        max_features = self.max_pool(x)
        
        # Process through dense layers
        avg_embedding = self.dense1(avg_features)
        max_embedding = self.dense2(max_features)
        
        # Combine embeddings
        combined = avg_embedding + max_embedding
        return self.dropout(combined, training=training)

# Build the Complete Model
def build_enhanced_geo_cnn(input_shape=(200, 200, 1), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)
    
    # Initial edge detection and feature extraction
    x = AdvancedEdgeLayer()(inputs)
    
    # First residual block with pooling
    x = GeometricResidualBlock(32)(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
    
    # Second residual block with pooling
    x = GeometricResidualBlock(64)(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
    
    # Apply spatial attention
    x = SpatialAttention()(x)
    
    # Third residual block
    x = GeometricResidualBlock(128)(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
    
    # Extract shape features
    x = ShapeFeatureExtractor(embedding_dim=128)(x)
    
    # Classification layers with higher capacity
    x = tf.keras.layers.Dense(128, activation='relu', 
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    
    # Final classification layer
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    
    return tf.keras.Model(inputs, outputs)

# --- STEP 4: Learning Rate Scheduling and Training ---

# Create learning rate schedule
def step_decay_schedule(epoch):
    initial_lr = 0.001
    if epoch < 5:
        return initial_lr  # Keep initial rate for first 5 epochs
    elif epoch < 15:
        return initial_lr * 0.1  # Reduce by factor of 10
    else:
        return initial_lr * 0.01  # Reduce by factor of 100

# Create callbacks
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(step_decay_schedule)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True
)

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_shape_detector.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Build and compile model
model = build_enhanced_geo_cnn(input_shape=(200, 200, 1), num_classes=num_classes)

# Mixed precision for faster training on compatible GPUs (commented out)
'''
# Enable mixed precision
from tensorflow.keras.mixed_precision import experimental as mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print('Compute dtype:', policy.compute_dtype)
print('Variable dtype:', policy.variable_dtype)
'''

# Compile with standard optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Model summary
model.summary()

# --- STEP 5: Training with TensorBoard Support ---
# Create TensorBoard callback (commented out)
'''
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir, 
    histogram_freq=1,
    write_graph=True,
    write_images=True,
    update_freq='epoch'
)

# Add tensorboard_callback to callbacks list below
'''

# Train with callbacks
epochs = 30
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[lr_scheduler, early_stopping, model_checkpoint]
)

# --- STEP 6: Evaluate on Test Set ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"✅ Test Accuracy: {test_acc:.4f}")

# --- STEP 7: Visualization and Analysis ---

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

# Function to get confusion matrix
def plot_confusion_matrix(model, test_ds, class_names):
    # Get predictions
    true_labels = []
    pred_labels = []
    
    for images, labels in test_ds:
        preds = model.predict(images)
        preds = np.argmax(preds, axis=1)
        
        true_labels.extend(labels.numpy())
        pred_labels.extend(preds)
    
    # Create confusion matrix
    cm = tf.math.confusion_matrix(true_labels, pred_labels)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    
    # Add labels
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    
    # Add counts
    thresh = cm.numpy().max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.savefig('confusion_matrix.png')
    plt.show()

# Plot confusion matrix after training
plot_confusion_matrix(model, test_ds, class_names)

# --- STEP 8: Save and Export Model ---
# Save the full model
model.save('complete_shape_detector_model')

# Convert to TensorFlow Lite format (commented out)
'''
# Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model
with open('shape_detector_model.tflite', 'wb') as f:
    f.write(tflite_model)
'''

# --- STEP 9: Function for Single Image Prediction ---
def predict_single_image(image_path, model, class_names):
    # Read and preprocess the image
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.rgb_to_grayscale(img)
    img = tf.image.resize(img, image_size)
    img = tf.cast(img, tf.float32) / 255.0
    
    # Add batch dimension
    img = tf.expand_dims(img, 0)
    
    # Make prediction
    pred = model.predict(img)
    pred_class = np.argmax(pred[0])
    confidence = pred[0][pred_class]
    
    # Display results
    print(f"Predicted class: {class_names[pred_class]}")
    print(f"Confidence: {confidence*100:.2f}%")
    
    # Display image with prediction
    plt.figure(figsize=(6, 6))
    plt.imshow(tf.squeeze(img).numpy(), cmap='gray')
    plt.title(f"Predicted: {class_names[pred_class]} ({confidence*100:.1f}%)")
    plt.axis('off')
    plt.show()
    
    return class_names[pred_class], confidence

# Example usage of single image prediction (commented out)
'''
# Example: Predict a single test image
test_image = test_paths[0]  # Get first test image path
predict_single_image(test_image, model, class_names)
'''

GPU Available:  []
Loading train data...
Loading validation data...
Loading test data...
Found 76 classes: ['Circle_Circle_Chord', 'Circle_Circle_Diameter', 'Circle_Circle_Plain', 'Circle_Circle_Radius', 'Inscribed_Circle_Square', 'Inscribed_Circle_Square_diagonals', 'Inscribed_Circle_Square_two_diagonals', 'Inscribed_Circle_right_angle_triangle', 'Inscribed_Circle_triangle', 'Inscribed_Square_Circle', 'Parallelogram_para_diag_perp_W', 'Parallelogram_para_diag_perp_Wo', 'Parallelogram_para_double_diagonal_W', 'Parallelogram_para_double_diagonal_Wo', 'Parallelogram_para_perp_W', 'Parallelogram_para_perp_Wo', 'Parallelogram_para_single_diagonal_W', 'Parallelogram_para_single_diagonal_Wo', 'Parallelogram_plain_para__W', 'Parallelogram_plain_para__Wo', 'Rectangle_plain_rect_W', 'Rectangle_plain_rect_Wo', 'Rectangle_rectangle_one_angle_one_diagonal_W', 'Rectangle_rectangle_one_angle_one_diagonal_Wo', 'Rectangle_rectangle_two_angles_one_diagonal_W', 'Rectangle_rectangle_two_angles_one_diagon

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 200, 200, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ advanced_edge_layer_1           │ (None, 200, 200, 16)   │           368 │
│ (AdvancedEdgeLayer)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ geometric_residual_block_3      │ (None, 200, 200, 32)   │        14,688 │
│ (GeometricResidualBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 100, 100, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ geometric_residual_block_4      │ (None, 100, 100, 64)   │        58,048 │
│ (GeometricResidualBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 50, 50, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_attention_1             │ (None, 50, 50, 64)     │        16,129 │
│ (SpatialAttention)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ geometric_residual_block_5      │ (None, 50, 50, 128)    │       230,784 │
│ (GeometricResidualBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 25, 25, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ shape_feature_extractor_1       │ (None, 128)            │        33,024 │
│ (ShapeFeatureExtractor)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 76)             │         9,804 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 379,357 (1.45 MB)

 Trainable params: 378,429 (1.44 MB)

 Non-trainable params: 928 (3.62 KB)

Epoch 1/30
  308/11448 ━━━━━━━━━━━━━━━━━━━━ 5:31:57 2s/step - accuracy: 0.9659 - loss: 0.2185

KeyboardInterrupt: 

In [22]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from glob import glob

# --- CONFIGURATION ---
data_root = r"C:\Users\ashut\Desktop\ElementaryCQT"
image_size = (200, 200)
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE

# --- STEP 1: Load All Images and Labels ---
def load_data():
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(data_root))
    class_map = {name: idx for idx, name in enumerate(class_names)}

    for shape_class in class_names:
        shape_dir = os.path.join(data_root, shape_class)
        for subtype in os.listdir(shape_dir):
            subtype_dir = os.path.join(shape_dir, subtype)
            if os.path.isdir(subtype_dir):
                for img_file in glob(f"{subtype_dir}/*.png") + glob(f"{subtype_dir}/*.jpg"):
                    image_paths.append(img_file)
                    labels.append(class_map[shape_class])

    return image_paths, labels, class_names

# --- STEP 2: Preprocess Images ---
def preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=1, expand_animations=False)  # convert to grayscale
    image.set_shape([None, None, 1])  # enforce shape rank
    image = tf.image.resize(image, image_size)  # resize to 200x200
    image = tf.cast(image, tf.float32) / 255.0  # normalize to [0, 1]
    return image, label

# --- STEP 3: Create Dataset Splits ---
image_paths, labels, class_names = load_data()
num_classes = len(class_names)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

def make_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)

print(f"✅ Dataset sizes — Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")


# --- Attention Layer ---
class ShapeAttention(tf.keras.layers.Layer):
    def call(self, x):
        attn = tf.reduce_mean(x, axis=-1, keepdims=True)
        flat_attn = tf.reshape(attn, [tf.shape(x)[0], -1])
        norm_attn = tf.nn.softmax(flat_attn, axis=-1)
        norm_attn = tf.reshape(norm_attn, tf.shape(attn))
        return x * norm_attn

# --- Convolutional Block ---
def conv_block(x, filters, kernel_size=3, pool=True):
    x = tf.keras.layers.Conv2D(filters, kernel_size, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2D(filters, kernel_size, padding='same', activation='relu')(x)
    if pool:
        x = tf.keras.layers.MaxPooling2D()(x)
    return x

# --- Build the Enhanced Model ---
def build_robust_geo_cnn(input_shape=(200, 200, 1), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)
    
    # Block 1: 64 filters + attention
    x = conv_block(inputs, 64)
    x = ShapeAttention()(x)
    
    # Block 2: 64 filters + attention
    x = conv_block(x, 64)
    x = ShapeAttention()(x)
    
    # Block 3: 128 filters + attention
    x = conv_block(x, 128)
    x = ShapeAttention()(x)
    
    # Block 4: 128 filters + attention
    x = conv_block(x, 128)
    x = ShapeAttention()(x)
    
    # Global Max Pooling
    x = tf.keras.layers.GlobalMaxPooling2D()(x)
    
    # Dense Layers
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    
    return tf.keras.Model(inputs, outputs)

# --- Compile and Train ---
model = build_robust_geo_cnn(input_shape=(200, 200, 1), num_classes=num_classes)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_ds, validation_data=val_ds, epochs=15)

# --- Evaluate ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"✅ Enhanced Test Accuracy: {test_acc:.4f}")


✅ Dataset sizes — Train: 707047, Val: 88381, Test: 88381
Epoch 1/15
17698/22096 ━━━━━━━━━━━━━━━━━━━━ 2:16:46 2s/step - accuracy: 0.4141 - loss: 1.9710

KeyboardInterrupt: 

In [19]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import os
import shutil
import matplotlib.pyplot as plt

# --- STEP 1: Data Preparation ---
main_dir = r"C:\Users\ashut\Desktop\ElementaryCQT"
train_dir = os.path.join(main_dir, 'train')
val_dir = os.path.join(main_dir, 'val')
test_dir = os.path.join(main_dir, 'test')

def split_dataset(source_dir, dest_dir_train, dest_dir_val, dest_dir_test, val_size=0.1, test_size=0.1):
    for shape_class in os.listdir(source_dir):
        shape_class_path = os.path.join(source_dir, shape_class)
        if os.path.isdir(shape_class_path) and shape_class not in ['train', 'val', 'test']:
            for subfolder in os.listdir(shape_class_path):
                subfolder_path = os.path.join(shape_class_path, subfolder)
                if os.path.isdir(subfolder_path):
                    all_images = [img for img in os.listdir(subfolder_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
                    if not all_images:
                        continue

                    train_imgs, test_imgs = train_test_split(all_images, test_size=test_size, random_state=42)
                    train_imgs, val_imgs = train_test_split(train_imgs, test_size=val_size, random_state=42)

                    class_name = f"{shape_class}_{subfolder}"
                    os.makedirs(os.path.join(dest_dir_train, class_name), exist_ok=True)
                    os.makedirs(os.path.join(dest_dir_val, class_name), exist_ok=True)
                    os.makedirs(os.path.join(dest_dir_test, class_name), exist_ok=True)

                    for img in train_imgs:
                        shutil.copy(os.path.join(subfolder_path, img), os.path.join(dest_dir_train, class_name, img))
                    for img in val_imgs:
                        shutil.copy(os.path.join(subfolder_path, img), os.path.join(dest_dir_val, class_name, img))
                    for img in test_imgs:
                        shutil.copy(os.path.join(subfolder_path, img), os.path.join(dest_dir_test, class_name, img))

split_dataset(main_dir, train_dir, val_dir, test_dir, val_size=0.1, test_size=0.1)

# --- STEP 2: Data Augmentation and Preprocessing ---
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_ds = train_datagen.flow_from_directory(
    train_dir,
    target_size=(200, 200),
    batch_size=32,
    class_mode='sparse')

val_ds = val_datagen.flow_from_directory(
    val_dir,
    target_size=(200, 200),
    batch_size=32,
    class_mode='sparse')

test_ds = test_datagen.flow_from_directory(
    test_dir,
    target_size=(200, 200),
    batch_size=32,
    class_mode='sparse')

print(f"\nClasses found: {list(train_ds.class_indices.keys())}")
print(f"Total training images: {train_ds.samples}")
print(f"Total validation images: {val_ds.samples}")
print(f"Total test images: {test_ds.samples}")

# --- STEP 3: CNN Model Architecture ---
def build_geo_cnn(num_classes):
    inputs = layers.Input(shape=(200, 200, 3))
    
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)

# --- STEP 4: Model Compilation ---
model = build_geo_cnn(num_classes=len(train_ds.class_indices))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# --- STEP 5: Train the Model ---
print("\n--- Starting Training ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    verbose=1  # Shows real-time training progress
)

# --- STEP 6: Evaluate on Test Set ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Loss: {test_loss}, Test Accuracy: {test_acc}")

# --- STEP 7: Save the Model ---
# model.save('geometric_cnn_model.h5')

# --- STEP 8: Visualize Training History ---
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()


Found 366335 images belonging to 76 classes.
Found 65334 images belonging to 76 classes.
Found 72140 images belonging to 76 classes.

Classes found: ['Circle_Circle_Chord', 'Circle_Circle_Diameter', 'Circle_Circle_Plain', 'Circle_Circle_Radius', 'Inscribed_Circle_Square', 'Inscribed_Circle_Square_diagonals', 'Inscribed_Circle_Square_two_diagonals', 'Inscribed_Circle_right_angle_triangle', 'Inscribed_Circle_triangle', 'Inscribed_Square_Circle', 'Parallelogram_para_diag_perp_W', 'Parallelogram_para_diag_perp_Wo', 'Parallelogram_para_double_diagonal_W', 'Parallelogram_para_double_diagonal_Wo', 'Parallelogram_para_perp_W', 'Parallelogram_para_perp_Wo', 'Parallelogram_para_single_diagonal_W', 'Parallelogram_para_single_diagonal_Wo', 'Parallelogram_plain_para__W', 'Parallelogram_plain_para__Wo', 'Rectangle_plain_rect_W', 'Rectangle_plain_rect_Wo', 'Rectangle_rectangle_one_angle_one_diagonal_W', 'Rectangle_rectangle_one_angle_one_diagonal_Wo', 'Rectangle_rectangle_two_angles_one_diagonal_W', 

KeyboardInterrupt: 

## 3 models

In [20]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, VGG19, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre
from sklearn.model_selection import train_test_split
import os
import shutil
import matplotlib.pyplot as plt

# --- STEP 1: Data Preparation ---
main_dir = r"C:\Users\ashut\Desktop\ElementaryCQT"
train_dir = os.path.join(main_dir, 'train')
val_dir = os.path.join(main_dir, 'val')
test_dir = os.path.join(main_dir, 'test')

def split_dataset(source_dir, dest_dir_train, dest_dir_val, dest_dir_test, val_size=0.1, test_size=0.1):
    for shape_class in os.listdir(source_dir):
        shape_class_path = os.path.join(source_dir, shape_class)
        if os.path.isdir(shape_class_path) and shape_class not in ['train', 'val', 'test']:
            for subfolder in os.listdir(shape_class_path):
                subfolder_path = os.path.join(shape_class_path, subfolder)
                if os.path.isdir(subfolder_path):
                    all_images = [img for img in os.listdir(subfolder_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
                    if not all_images:
                        continue

                    train_imgs, test_imgs = train_test_split(all_images, test_size=test_size, random_state=42)
                    train_imgs, val_imgs = train_test_split(train_imgs, test_size=val_size, random_state=42)

                    class_name = f"{shape_class}_{subfolder}"
                    os.makedirs(os.path.join(dest_dir_train, class_name), exist_ok=True)
                    os.makedirs(os.path.join(dest_dir_val, class_name), exist_ok=True)
                    os.makedirs(os.path.join(dest_dir_test, class_name), exist_ok=True)

                    for img in train_imgs:
                        shutil.copy(os.path.join(subfolder_path, img), os.path.join(dest_dir_train, class_name, img))
                    for img in val_imgs:
                        shutil.copy(os.path.join(subfolder_path, img), os.path.join(dest_dir_val, class_name, img))
                    for img in test_imgs:
                        shutil.copy(os.path.join(subfolder_path, img), os.path.join(dest_dir_test, class_name, img))

split_dataset(main_dir, train_dir, val_dir, test_dir, val_size=0.1, test_size=0.1)

# --- STEP 2: Data Loading ---
BATCH_SIZE = 32
IMG_SIZE = (224, 224)  # Use 224x224 for pretrained models

train_ds = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=vgg_pre).flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='sparse')
val_ds = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=vgg_pre).flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='sparse')
test_ds = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=vgg_pre).flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='sparse')

num_classes = len(train_ds.class_indices)
print(f"\nClasses found: {list(train_ds.class_indices.keys())}")

# --- STEP 3: Model Builders ---
def build_transfer_model(base_model, input_shape=(224, 224, 3), num_classes=10):
    base_model.trainable = False  # Freeze base layers
    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

models_to_train = {
    "VGG16": build_transfer_model(VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3)), num_classes=num_classes),
    "VGG19": build_transfer_model(VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3)), num_classes=num_classes),
    "ResNet50": build_transfer_model(ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3)), num_classes=num_classes),
}

results = {}

# --- STEP 4: Train and Evaluate Each Model ---
for name, model in models_to_train.items():
    print(f"\n--- Training {name} ---")
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,  # Set to a low number for quicker testing
        verbose=1
    )

    test_loss, test_acc = model.evaluate(test_ds)
    print(f"{name} Test Accuracy: {test_acc:.4f}")
    results[name] = {
        'model': model,
        'accuracy': test_acc,
        'loss': test_loss,
        'history': history
    }

# --- STEP 5: Plot Accuracy Comparison ---
plt.figure(figsize=(10, 6))
for name, data in results.items():
    plt.plot(data['history'].history['val_accuracy'], label=f"{name} Val Acc")
plt.title("Validation Accuracy per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


Found 366335 images belonging to 76 classes.
Found 65334 images belonging to 76 classes.
Found 72140 images belonging to 76 classes.

Classes found: ['Circle_Circle_Chord', 'Circle_Circle_Diameter', 'Circle_Circle_Plain', 'Circle_Circle_Radius', 'Inscribed_Circle_Square', 'Inscribed_Circle_Square_diagonals', 'Inscribed_Circle_Square_two_diagonals', 'Inscribed_Circle_right_angle_triangle', 'Inscribed_Circle_triangle', 'Inscribed_Square_Circle', 'Parallelogram_para_diag_perp_W', 'Parallelogram_para_diag_perp_Wo', 'Parallelogram_para_double_diagonal_W', 'Parallelogram_para_double_diagonal_Wo', 'Parallelogram_para_perp_W', 'Parallelogram_para_perp_Wo', 'Parallelogram_para_single_diagonal_W', 'Parallelogram_para_single_diagonal_Wo', 'Parallelogram_plain_para__W', 'Parallelogram_plain_para__Wo', 'Rectangle_plain_rect_W', 'Rectangle_plain_rect_Wo', 'Rectangle_rectangle_one_angle_one_diagonal_W', 'Rectangle_rectangle_one_angle_one_diagonal_Wo', 'Rectangle_rectangle_two_angles_one_diagonal_W', 

KeyboardInterrupt: 